# CSA08 Assignment 4 – Robust Data Processing with Files, Exceptions, Modules and Packages

**Student Name:** Darshan A  
**Register Number:** 1973260011  
**Course Code:** CSA0812  
**Task 3:** Design the Error Handling: Exceptions in the Attendance Processor  
**Marks:** 9 | **Bloom’s Level:** K4 – Analyse

---

## Aim
To identify possible failures in an attendance file processor and implement suitable exception handling using built-in and custom exceptions.

## Objectives
- Identify file and record-processing failures and their corresponding exceptions.
- Define a custom `InvalidRecord` exception.
- Use `try`, specific `except` clauses, `else` and `finally`.
- Preserve the original cause using `raise ... from ...`.
- Demonstrate why silently ignoring all exceptions is unsafe.


## Task 3.1 – Identify the Failures (3 Marks)

| Failure | Possible cause | Exception / handling |
|---|---|---|
| Missing folder or file | The input path does not exist. | `FileNotFoundError` |
| Permission denied | The program does not have permission to read the file. | `PermissionError` |
| Wrong text encoding | File bytes cannot be decoded using the selected encoding. | `UnicodeDecodeError` |
| Damaged record | A line has missing fields or an invalid attendance status. | Custom `InvalidRecord` |
| Invalid value | A value cannot be converted or does not meet the required rule. | `ValueError` |
| Missing field accessed by index | The program tries to access a field that is not present. | `IndexError` |
| Empty file | The file contains no records. | No exception is raised; the program must handle zero records as a valid empty result. |
| Path refers to a folder | A directory is supplied where a file is expected. | `IsADirectoryError` |

**Explanation:** Specific exceptions help the program respond to each problem correctly. A damaged attendance line should be recorded and skipped, while a missing file or permission problem should be reported as a file-access failure.


## Task 3.2 – Design a Custom Exception (2 Marks)

A custom exception named `InvalidRecord` is used when an attendance line does not contain exactly three fields or its status is not `P` or `A`.

- **Where it is raised:** Inside `parse_line()`, immediately after validating the line.
- **Where it is handled:** Inside the loop in `summarise()`, so one damaged line is recorded and the next line can still be processed.

This separates record-validation errors from file-access errors.


In [1]:
# Define a custom exception and test the line parser

class InvalidRecord(Exception):
    """Raised when an attendance line is not in the expected format."""


def parse_line(line):
    parts = line.strip().split(",")

    if len(parts) != 3:
        raise InvalidRecord(
            f"expected 3 fields, found {len(parts)}"
        )

    register_number = parts[0].strip()
    date = parts[1].strip()
    status = parts[2].strip()

    if status not in ("P", "A"):
        raise InvalidRecord(f"invalid status '{status}'")

    if not register_number or not date:
        raise InvalidRecord("register number or date is missing")

    return register_number, date, status


# Test one valid line
print(parse_line("192311001,2026-10-08,P"))

# Test one invalid line
try:
    parse_line("192311002,2026-10-08,L")
except InvalidRecord as error:
    print("Invalid record:", error)


('192311001', '2026-10-08', 'P')
Invalid record: invalid status 'L'


## Task 3.3 – Use the Complete try Statement (3 Marks)

The example below demonstrates:
- `try` – attempts to open and process the file.
- Specific `except` clauses – handle missing files, permission problems, decoding errors and invalid records.
- `else` – runs when the file is processed without a file-level error.
- `finally` – runs whether processing succeeds or fails.
- `raise ... from ...` – raises a clear application-level error while preserving the original exception as its cause.


In [2]:
from pathlib import Path


class AttendanceFileError(Exception):
    """Raised when an attendance file cannot be processed."""


def read_attendance_file(filename):
    valid_records = []
    damaged_records = []

    try:
        with open(filename, "r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                if not line.strip():
                    continue

                try:
                    record = parse_line(line)
                    valid_records.append(record)

                except InvalidRecord as error:
                    damaged_records.append(
                        (line_number, line.rstrip(), str(error))
                    )

    except FileNotFoundError as error:
        raise AttendanceFileError(
            f"Attendance file not found: {filename}"
        ) from error

    except PermissionError as error:
        raise AttendanceFileError(
            f"Permission denied while reading: {filename}"
        ) from error

    except UnicodeDecodeError as error:
        raise AttendanceFileError(
            f"Could not decode attendance file: {filename}"
        ) from error

    else:
        print("File was opened and processed successfully.")

    finally:
        print("File processing attempt finished.")

    return valid_records, damaged_records


# Create a small test file
with open("attendance_try_demo.txt", "w", encoding="utf-8") as file:
    file.write(
        "192311001,2026-10-08,P\n"
        "192311002,2026-10-08,A\n"
        "192311003,2026-10-08,L\n"
    )

# Run the processor
try:
    valid, damaged = read_attendance_file("attendance_try_demo.txt")
    print("Valid records:", valid)
    print("Damaged records:", damaged)

except AttendanceFileError as error:
    print("File error:", error)


File was opened and processed successfully.
File processing attempt finished.
Valid records: [('192311001', '2026-10-08', 'P'), ('192311002', '2026-10-08', 'A')]
Damaged records: [(3, '192311003,2026-10-08,L', "invalid status 'L'")]


### Demonstrate `raise ... from ...` with a Missing File

The next cell uses a filename that does not exist. The original `FileNotFoundError` is wrapped in `AttendanceFileError`. The `__cause__` attribute confirms that the original exception has been preserved.


In [3]:
try:
    read_attendance_file("file_that_does_not_exist.txt")

except AttendanceFileError as error:
    print("Handled error:", error)
    print("Original cause:", type(error.__cause__).__name__)


File processing attempt finished.
Handled error: Attendance file not found: file_that_does_not_exist.txt
Original cause: FileNotFoundError


## Complete the Code – TODO 1 and TODO 2

The following program follows the assignment's supplied structure. The two TODO sections are completed. It parses each line, counts present and absent records, and records invalid lines with their line numbers.

The expected output is:

```text
(1, 1, [(3, 'expected 3 fields, found 2'), (4, "invalid status 'L'")])
```


In [4]:
import os
import tempfile


class InvalidRecord(Exception):
    """Raised when a line of the attendance file is not in the expected format."""


def parse_line(line):
    """'192311001,2026-10-08,P' -> ('192311001', '2026-10-08', 'P')"""
    parts = line.strip().split(",")

    # TODO 1 completed: validate the fields and return a tuple
    if len(parts) != 3:
        raise InvalidRecord(
            f"expected 3 fields, found {len(parts)}"
        )

    if parts[2].strip() not in ("P", "A"):
        raise InvalidRecord(f"invalid status '{parts[2].strip()}'")

    return parts[0].strip(), parts[1].strip(), parts[2].strip()


def summarise(path):
    present, absent, errors = 0, 0, []

    # TODO 2 completed: open safely, process lines and record invalid lines
    try:
        with open(path, "r", encoding="utf-8") as file:
            for line_number, line in enumerate(file, start=1):
                if not line.strip():
                    continue

                try:
                    register_number, date, status = parse_line(line)

                    if status == "P":
                        present += 1
                    else:
                        absent += 1

                except InvalidRecord as error:
                    errors.append((line_number, str(error)))

    except FileNotFoundError as error:
        print("File not found:", path)
        return present, absent, errors

    return present, absent, errors


data = (
    "192311001,2026-10-08,P\n192311002,2026-10-08,A\n"
    "192311003,2026-10-08\n192311004,2026-10-08,L\n"
)

with tempfile.NamedTemporaryFile(
    "w", suffix=".csv", delete=False, encoding="utf-8"
) as tmp:
    tmp.write(data)
    temporary_filename = tmp.name

print(summarise(temporary_filename))
os.remove(temporary_filename)


(1, 1, [(3, 'expected 3 fields, found 2'), (4, "invalid status 'L'")])


## Handle a File That Does Not Exist in `summarise()`

The assignment asks for a missing-file test. The `summarise()` function catches `FileNotFoundError`, displays a useful message and returns safely instead of terminating the program.


In [5]:
# Test summarise() with a filename that does not exist

missing_result = summarise("attendance_file_missing.csv")
print("Returned result:", missing_result)


File not found: attendance_file_missing.csv
Returned result: (0, 0, [])


## Task 3.4 – Misuse Experiment (1 Mark)

A broad handler such as `except Exception: pass` catches almost every ordinary program error and silently ignores it. The next experiment deliberately uses a misspelled variable name to create a `NameError`. Because the broad handler ignores the error, no useful error message is displayed.


In [6]:
# Demonstrate why a broad exception handler is dangerous

def demonstrate_hidden_error():
    try:
        total_present = 5
        print(totel_present)  # Deliberate typing mistake: variable name is misspelled

    except Exception:
        pass

    print("Program continued, but the error was hidden.")


demonstrate_hidden_error()


Program continued, but the error was hidden.


### Misuse Experiment – Observation

**Observed output:**

```text
Program continued, but the error was hidden.
```

The variable `totel_present` does not exist, so Python raises a `NameError`. The broad `except Exception: pass` catches the error and does nothing. As a result, the programmer does not see the error message and may incorrectly believe the calculation worked.

**Conclusion:** Specific exception handlers are safer because they handle expected errors clearly and allow unexpected programming mistakes to remain visible.


## Alternative Concept and Complexity Analysis

The current processor uses the **EAFP (Easier to Ask Forgiveness than Permission)** approach: it attempts to parse each line and catches `InvalidRecord` if the line is damaged.

An alternative is **LBYL (Look Before You Leap)**. In this approach, the program checks the number of fields and status using `if` statements before processing the record.

### LBYL example

```python
parts = line.strip().split(",")

if len(parts) == 3 and parts[2].strip() in ("P", "A"):
    # Process the valid record
    pass
else:
    # Record the damaged line
    pass
```

| Situation | EAFP – try/except | LBYL – if checks |
|---|---|---|
| Bad lines are rare | Usually efficient because exceptions are uncommon. | Performs validation checks for every line. |
| Bad lines are common | Repeated exceptions can add processing overhead. | Often preferable because invalid data is identified through normal conditions. |
| Time complexity | O(n) | O(n) |
| Extra working space | O(1) per line, excluding stored errors | O(1) per line, excluding stored errors |

Here, `n` is the number of lines. Both approaches examine each line once, so both have O(n) time complexity. Their practical running time can differ because raising and catching exceptions has additional overhead.

**Conclusion:** EAFP is suitable when damaged lines are rare and parsing is attempted directly. LBYL can be preferable when invalid lines are common and simple validation checks can identify them before parsing.


## Task 3 – Conclusion

The attendance processor uses a custom `InvalidRecord` exception to identify damaged lines and specific built-in exceptions to handle file-access and decoding failures. The `try`, `except`, `else` and `finally` blocks provide structured error handling, while `raise ... from ...` preserves the original cause of a file-processing error. The misuse experiment demonstrates that silently ignoring all exceptions can hide programming mistakes. Specific exception handling makes the program easier to debug and more reliable.
